# Huber + Hilbert + Direct Analog Gate Submission

Self-contained Kaggle submission notebook for the current Huber + additive Hilbert residual model plus log-plus-trajectory nearest-analog correction. The analog layer is recomputed inside the notebook from training wells only and keeps internet disabled.

## 1. Setup

In [1]:
"Imports, paths, constants, and data discovery."

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

KAGGLE_ROOT = Path("/kaggle")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")

MODEL_NAME = "huber_hilbert_direct_analog_gate"
MAX_TARGET_ROWS_PER_WELL = 140
NU = 3.0
HSGP_PRIOR_SCALE = 0.55
INTERCEPT_PRIOR_SCALE = 1.0
SIGMA_PRIOR_SCALE = 1.5
MAX_OPT_ITER = 1000
HSGP_M = 8
HSGP_ELL = 3.0
HUBER_ALPHA = 0.0001
HUBER_EPSILON = 1.35
HUBER_MAX_ITER = 1200
ANALOG_OFFSET_CLIP = 12.0
ANALOG_PROGRESS_SHRINK = {"z0": 0.025, "z1": 0.025, "z2": 0.075, "z3": 0.100}
ANALOG_FEATURE_COLUMNS = [
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "known_tvt_span",
    "xy_span_known",
    "z_drop_known",
]
ANALOG_LAGS = np.arange(-40.0, 42.5, 2.5)

FEATURE_COLUMNS = [
    "distance_from_boundary_md",
    "eval_progress",
    "MD",
    "Z",
    "md_norm",
    "dX_from_boundary",
    "dY_from_boundary",
    "dZ_from_boundary",
    "horizontal_dist_from_boundary",
    "tvt_slope_last_10",
    "tvt_slope_last_25",
    "tvt_slope_last_50",
    "tvt_slope_last_100",
    "z_slope_last_25",
    "z_slope_last_50",
    "z_slope_last_100",
    "tvt_slope_last_10_projected_delta",
    "tvt_slope_last_25_projected_delta",
    "tvt_slope_last_50_projected_delta",
    "tvt_slope_last_100_projected_delta",
    "last_known_tvt",
]
HSGP_COORDINATE_COLUMNS = [
    "distance_log1p",
    "eval_progress",
    "abs_projected_delta_10",
]


def local_data_candidates():
    candidates = [Path.cwd() / "data" / "raw"]
    candidates.extend(parent / "data" / "raw" for parent in Path.cwd().parents)
    return candidates


def resolve_base_dir():
    known = Path("/kaggle/input/rogii-wellbore-geology-prediction")
    for candidate in [known] + local_data_candidates():
        if (candidate / "sample_submission.csv").exists() and (candidate / "train").exists() and (candidate / "test").exists():
            return candidate
    if KAGGLE_INPUT_ROOT.exists():
        for sample_path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv"):
            candidate = sample_path.parent
            if (candidate / "train").exists() and (candidate / "test").exists():
                return candidate
        children = sorted(str(path) for path in KAGGLE_INPUT_ROOT.glob("*"))
        raise FileNotFoundError(f"Could not find competition data under /kaggle/input. Children: {children}")
    raise FileNotFoundError("Could not find Kaggle or local competition data.")


BASE_DIR = resolve_base_dir()

if KAGGLE_ROOT.exists():
    OUTPUT_PATH = KAGGLE_OUTPUT / "submission.csv"
else:
    LOCAL_ROOT = BASE_DIR.parent.parent
    OUTPUT_PATH = LOCAL_ROOT / "artifacts" / "reports" / f"submission__huber_hilbert_analog__{date.today().isoformat()}.csv"

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
SAMPLE_PATH = BASE_DIR / "sample_submission.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)

BASE_DIR: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw
OUTPUT_PATH: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\submission__huber_hilbert_analog__2026-06-23.csv


## 2. Helpers

In [2]:
"Loaders, slope-core features, additive Hilbert basis, and Huber residual MAP utilities."


def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def load_horizontal_well(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index
    return df


def load_sample_submission(path):
    sample = pd.read_csv(path)
    split = sample["id"].str.rsplit("_", n=1, expand=True)
    sample["wellname"] = split[0]
    sample["row_idx"] = split[1].astype(int)
    return sample


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def safe_corr(left, right):
    a = np.asarray(left, dtype=float)
    b = np.asarray(right, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if int(mask.sum()) < 5:
        return np.nan
    if float(np.nanstd(a[mask])) < 1e-9 or float(np.nanstd(b[mask])) < 1e-9:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def linear_slope(x_values, y_values):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    coef = np.polyfit(x[mask], y[mask], 1)
    return float(coef[0])


def interp_typewell_gr(typewell, tvt_values, lag=0.0):
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float) + lag, clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_analog_descriptor(well, directory):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    typewell_path = directory / f"{well}__typewell.csv"
    typewell = pd.read_csv(typewell_path) if typewell_path.exists() else pd.DataFrame({"TVT": [], "GR": []})
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    if len(known) == 0:
        return {"wellname": well, **{col: np.nan for col in ANALOG_FEATURE_COLUMNS}, "last_known_tvt": np.nan}
    tail = known.tail(300).copy()
    descriptor = {"wellname": well}
    descriptor["known_gr_mean_last300"] = float(tail["GR"].mean()) if len(tail) else np.nan
    descriptor["known_gr_std_last300"] = float(tail["GR"].std()) if len(tail) else np.nan
    descriptor["known_gr_slope_last300"] = linear_slope(tail["TVT_input"], tail["GR"])
    descriptor["known_tvt_span"] = float(known["TVT_input"].max() - known["TVT_input"].min()) if known["TVT_input"].notna().any() else np.nan
    descriptor["xy_span_known"] = float(np.hypot(known["X"].max() - known["X"].min(), known["Y"].max() - known["Y"].min())) if len(known) else np.nan
    descriptor["z_drop_known"] = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else np.nan
    descriptor["last_known_tvt"] = float(known["TVT_input"].iloc[-1])
    if len(tail) >= 5 and len(typewell) >= 5:
        lag_rows = []
        for lag in ANALOG_LAGS:
            shifted_gr = interp_typewell_gr(typewell, tail["TVT_input"].to_numpy(), lag=lag)
            corr = safe_corr(tail["GR"], shifted_gr)
            lag_rows.append((lag, corr))
        lag_frame = pd.DataFrame(lag_rows, columns=["lag", "corr"])
        best = lag_frame.sort_values("corr", ascending=False, na_position="last").iloc[0]
        descriptor["typewell_best_lag_corr"] = float(best["corr"])
        descriptor["typewell_best_lag_tvt"] = float(best["lag"])
    else:
        descriptor["typewell_best_lag_corr"] = np.nan
        descriptor["typewell_best_lag_tvt"] = np.nan
    return descriptor


def fit_analog_reference(train_df, train_directory):
    wells = sorted(train_df["wellname"].unique())
    descriptors = pd.DataFrame([build_analog_descriptor(well, train_directory) for well in wells])
    medians = descriptors[ANALOG_FEATURE_COLUMNS].median(numeric_only=True)
    scale = descriptors[ANALOG_FEATURE_COLUMNS].std(numeric_only=True).replace(0, np.nan).fillna(1.0)
    scaled = (descriptors[ANALOG_FEATURE_COLUMNS].fillna(medians) - medians) / scale
    curves = {}
    for well, group in train_df.groupby("wellname"):
        curve = group[["eval_progress", "delta_tvt_from_boundary"]].dropna().sort_values("eval_progress")
        if len(curve) >= 2:
            curves[well] = curve.groupby("eval_progress", as_index=False)["delta_tvt_from_boundary"].mean()
    return {"descriptors": descriptors, "medians": medians, "scale": scale, "scaled": scaled.to_numpy(dtype=float), "wells": descriptors["wellname"].to_numpy(), "curves": curves}


def analog_progress_zone(progress):
    p = float(progress)
    if p <= 0.25:
        return "z0"
    if p <= 0.50:
        return "z1"
    if p <= 0.75:
        return "z2"
    return "z3"


def select_analog_well(reference, query_descriptor, query_well):
    query = pd.DataFrame([query_descriptor])
    query_scaled = ((query[ANALOG_FEATURE_COLUMNS].fillna(reference["medians"]) - reference["medians"]) / reference["scale"]).to_numpy(dtype=float)[0]
    distances = np.sqrt(((reference["scaled"] - query_scaled) ** 2).mean(axis=1))
    same = reference["wells"] == query_well
    if bool(same.any()) and int((~same).sum()) > 0:
        distances = distances.copy()
        distances[same] = np.inf
    order = np.argsort(distances)
    for pos in order:
        well = str(reference["wells"][pos])
        if well in reference["curves"]:
            return well, float(distances[pos])
    return None, np.nan


def interpolate_analog_delta(reference, analog_well, progress_values):
    if analog_well is None or analog_well not in reference["curves"]:
        return np.zeros(len(progress_values), dtype=float)
    curve = reference["curves"][analog_well]
    return np.interp(np.asarray(progress_values, dtype=float), curve["eval_progress"].to_numpy(), curve["delta_tvt_from_boundary"].to_numpy(), left=curve["delta_tvt_from_boundary"].iloc[0], right=curve["delta_tvt_from_boundary"].iloc[-1])


def analog_gate_correction(reference, well, directory, frame, predicted_delta):
    descriptor = build_analog_descriptor(well, directory)
    analog_well, distance = select_analog_well(reference, descriptor, well)
    analog_delta = interpolate_analog_delta(reference, analog_well, frame["eval_progress"].to_numpy())
    analog_gap = analog_delta - np.asarray(predicted_delta, dtype=float)
    shrink = np.array([ANALOG_PROGRESS_SHRINK[analog_progress_zone(value)] for value in frame["eval_progress"].to_numpy()], dtype=float)
    correction = np.clip(analog_gap * shrink, -ANALOG_OFFSET_CLIP, ANALOG_OFFSET_CLIP)
    return correction, analog_well, distance


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, row_indices=None, max_train_rows=None, require_target=False):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    known_mask = horizontal["TVT_input"].notna()
    if require_target:
        candidate_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    else:
        candidate_mask = horizontal["TVT_input"].isna()
    if row_indices is not None:
        row_index_set = set(int(idx) for idx in row_indices)
        candidate_mask = candidate_mask & horizontal["row_idx"].isin(row_index_set)
    if int(candidate_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_x = float(x.iloc[last_known_pos])
    last_known_y = float(y.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}

    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    if max_train_rows is None:
        target_positions = np.flatnonzero(candidate_mask.to_numpy())
    else:
        target_positions = deterministic_sample_positions(candidate_mask, max_train_rows)
    all_target_positions = np.flatnonzero(candidate_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        tvt_slope_10 = tvt_slopes["tvt_slope_last_10"]
        projected_delta_10 = tvt_slope_10 * distance_md if np.isfinite(tvt_slope_10) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "distance_log1p": float(np.log1p(max(distance_md, 0.0))),
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "dX_from_boundary": float(x.iloc[pos] - last_known_x),
            "dY_from_boundary": float(y.iloc[pos] - last_known_y),
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "abs_projected_delta_10": float(abs(projected_delta_10)) if np.isfinite(projected_delta_10) else np.nan,
        }
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        if require_target:
            row["TVT"] = float(horizontal["TVT"].iloc[pos])
            row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        rows.append(row)
    return pd.DataFrame(rows)


def make_huber_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", HuberRegressor(alpha=HUBER_ALPHA, epsilon=HUBER_EPSILON, max_iter=HUBER_MAX_ITER)),
    ])


def fit_preprocessor(frame, columns):
    medians = frame[columns].median(numeric_only=True)
    filled = frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_preprocessor(frame, columns, medians, means, stds):
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def hilbert_basis_1d(values, m=HSGP_M, ell=HSGP_ELL):
    clipped = np.clip(values, -ell, ell)
    shifted = (clipped + ell) / (2.0 * ell)
    pieces = []
    for j in range(1, m + 1):
        pieces.append(np.sqrt(1.0 / ell) * np.sin(np.pi * j * shifted))
    return np.column_stack(pieces)


def additive_hilbert_features(frame, model):
    coords = apply_preprocessor(frame, HSGP_COORDINATE_COLUMNS, model["coord_medians"], model["coord_means"], model["coord_stds"])
    blocks = [hilbert_basis_1d(coords[:, idx]) for idx in range(coords.shape[1])]
    return np.column_stack(blocks)


def softplus(value):
    return np.logaddexp(0.0, value)


def residual_student_t_objective(params, X, y):
    intercept = params[0]
    beta = params[1:-1]
    sigma = softplus(params[-1]) + 1e-6
    residual = y - (intercept + X @ beta)
    scaled = residual / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    beta_prior = 0.5 * np.sum((beta / HSGP_PRIOR_SCALE) ** 2)
    intercept_prior = 0.5 * (intercept / INTERCEPT_PRIOR_SCALE) ** 2
    sigma_prior = 0.5 * (sigma / SIGMA_PRIOR_SCALE) ** 2
    return float(nll + beta_prior + intercept_prior + sigma_prior)


def fit_huber_hsgp_residual_model(train_df):
    huber = make_huber_model()
    y_delta = train_df["delta_tvt_from_boundary"].to_numpy(float)
    huber.fit(train_df[FEATURE_COLUMNS], y_delta)
    huber_pred = huber.predict(train_df[FEATURE_COLUMNS])
    residual_raw = y_delta - huber_pred
    coord_medians, coord_means, coord_stds = fit_preprocessor(train_df, HSGP_COORDINATE_COLUMNS)
    model = {"huber": huber, "coord_medians": coord_medians, "coord_means": coord_means, "coord_stds": coord_stds}
    X = additive_hilbert_features(train_df, model)
    y_mean = float(np.mean(residual_raw))
    y_std = float(np.std(residual_raw) + 1e-9)
    y = (residual_raw - y_mean) / y_std
    start = np.zeros(X.shape[1] + 2, dtype=float)
    start[-1] = np.log(np.expm1(1.0))
    result = minimize(
        residual_student_t_objective,
        start,
        args=(X, y),
        method="L-BFGS-B",
        options={"maxiter": MAX_OPT_ITER, "ftol": 1e-8, "maxls": 50},
    )
    if not result.success:
        print("optimizer_warning:", result.message)
    model.update({"params": result.x, "y_mean": y_mean, "y_std": y_std, "loss": float(result.fun), "success": bool(result.success), "nit": int(result.nit), "n_features": int(X.shape[1])})
    return model


def predict_delta(model, frame):
    huber_delta = model["huber"].predict(frame[FEATURE_COLUMNS])
    X = additive_hilbert_features(frame, model)
    params = model["params"]
    residual_scaled = params[0] + X @ params[1:-1]
    residual_delta = residual_scaled * model["y_std"] + model["y_mean"]
    return huber_delta + residual_delta


## 3. Train Huber + Additive HSGP Residual

In [3]:
"Build sampled target-zone training rows and fit Huber plus additive Hilbert residual." 

train_frames = []
train_wells = discover_wells(TRAIN_DIR)
for well in train_wells:
    frame = build_slope_core_features_for_well(
        well,
        TRAIN_DIR,
        max_train_rows=MAX_TARGET_ROWS_PER_WELL,
        require_target=True,
    )
    if not frame.empty:
        train_frames.append(frame)

train_df = pd.concat(train_frames, ignore_index=True)
model = fit_huber_hsgp_residual_model(train_df)
analog_reference = fit_analog_reference(train_df, TRAIN_DIR)
sigma_scaled = softplus(model["params"][-1]) + 1e-6

print("train_wells:", len(train_wells))
print("train_rows:", len(train_df))
print("model:", MODEL_NAME)
print("config:", {"huber_alpha": HUBER_ALPHA, "huber_epsilon": HUBER_EPSILON, "nu": NU, "hsgp_m": HSGP_M, "hsgp_ell": HSGP_ELL, "coordinates": HSGP_COORDINATE_COLUMNS, "max_iter": MAX_OPT_ITER, "optimizer_success": model["success"], "optimizer_iterations": model["nit"], "loss": round(model["loss"], 3), "sigma_scaled": round(float(sigma_scaled), 4), "n_features": model["n_features"], "analog_features": ANALOG_FEATURE_COLUMNS, "analog_wells": len(analog_reference["wells"])})

train_wells: 773
train_rows: 108220
model: huber_hilbert_direct_analog_gate
config: {'huber_alpha': 0.0001, 'huber_epsilon': 1.35, 'nu': 3.0, 'hsgp_m': 8, 'hsgp_ell': 3.0, 'coordinates': ['distance_log1p', 'eval_progress', 'abs_projected_delta_10'], 'max_iter': 1000, 'optimizer_success': True, 'optimizer_iterations': 157, 'loss': 35970.135, 'sigma_scaled': 0.6473, 'n_features': 24, 'analog_features': ['known_gr_mean_last300', 'known_gr_std_last300', 'known_gr_slope_last300', 'typewell_best_lag_corr', 'typewell_best_lag_tvt', 'known_tvt_span', 'xy_span_known', 'z_drop_known'], 'analog_wells': 773}


## 4. Write Submission

In [4]:
"Generate predictions in sample_submission order and write the CSV."

sample = load_sample_submission(SAMPLE_PATH)
predictions = {}

for well, group in sample.groupby("wellname", sort=True):
    frame = build_slope_core_features_for_well(
        well,
        TEST_DIR,
        row_indices=group["row_idx"].tolist(),
        max_train_rows=None,
        require_target=False,
    )
    if frame.empty:
        continue
    pred_delta = predict_delta(model, frame)
    analog_correction, analog_well, analog_distance = analog_gate_correction(analog_reference, well, TEST_DIR, frame, pred_delta)
    pred_tvt = frame["last_known_tvt"].to_numpy(float) + pred_delta + analog_correction
    for row_idx, value in zip(frame["row_idx"].astype(int), pred_tvt):
        predictions[f"{well}_{row_idx}"] = float(value)

submission = sample[["id"]].copy()
submission["tvt"] = submission["id"].map(predictions)

assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["tvt"].notna().all()
assert np.isfinite(submission["tvt"].to_numpy()).all()

submission.to_csv(OUTPUT_PATH, index=False)

print("submission_rows:", len(submission))
print("submission_path:", OUTPUT_PATH)
submission.head()

submission_rows: 14151
submission_path: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\submission__huber_hilbert_analog__2026-06-23.csv


,id,tvt
0,000d7d20_1442,11744.833936
1,000d7d20_1443,11744.852650
2,000d7d20_1444,11744.871339
3,000d7d20_1445,11744.889982
4,000d7d20_1446,11744.908619
